In [0]:
%run /Configurations/Init_Scripts

##Declaring the source and destination paths

In [0]:
# dbutils.widgets.text("PromotionName", "4P Beta")
# PromotionName = dbutils.widgets.get("PromotionName")

dbutils.widgets.text("job_id","-1")
JobId = dbutils.widgets.get("job_id")

dbutils.widgets.text("run_id","-1")
PipelineRunId = dbutils.widgets.get("run_id")

ConfigID=dbutils.widgets.text("ConfigID","44")
ConfigID=dbutils.widgets.get("ConfigID")

dbutils.widgets.text('sourceTypeId','2')
sourceTypeId = dbutils.widgets.get('sourceTypeId')

dbutils.widgets.text('CreatedBy','ADB_InvoiceException')
CreatedBy = dbutils.widgets.get('CreatedBy')

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")

dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
RawMaterialCodeFilePath=ExternalLocation_raw+'/MasterData/SmartCardMaterialCd/SmartCard_MaterialCode.csv'

dbutils.widgets.text("ExternalLocationName_silver", "/mntprod_silver")
ExternalLocationName_silver = dbutils.widgets.get("ExternalLocationName_silver")
SilverFilePath_ProductHierarchy=ExternalLocationName_silver+'/DIMProductHierarchy'
SilverFilePath_EquipmentMaster=ExternalLocationName_silver+'/DIMEquipmentMaster'
SilverFilePath_Promotion=ExternalLocationName_silver+'/DIMPromotion'
SilverFilePath_Smartcard=ExternalLocationName_silver+'/DIMSmartCard'

In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

In [0]:
# SilverFilePath_ProductHierarchy = '/mnt/silver/DIMProductHierarchy'
# SilverFilePath_EquipmentMaster = '/mnt/silver/DIMEquipmentMaster'
# SilverFilePath_Promotion = '/mnt/silver/DIMPromotion'
# SilverFilePath_Smartcard= '/mnt/silver/DIMSmartCard'

# RawMaterialCodeFilePath =  '/mnt/raw/MasterData/SmartCardMaterialCd/SmartCard_MaterialCode.csv'

In [0]:
# Read Equipment Master from silver zone
df_EquipmentMaster = (spark.read.format('delta').load(SilverFilePath_EquipmentMaster))\
                     .select('SerialNumberNbr','MaterialCd').dropDuplicates()

# Read Product Hierarchy from silver zone
df_ProdHrchy = (spark.read.format('delta')
                          .load(SilverFilePath_ProductHierarchy)
                          .select('ProductCd', 'ProductDesc')
                          .withColumnRenamed('ProductCd','MaterialCd')                
                          .withColumnRenamed('ProductDesc','MaterialDescription')         
                          .dropDuplicates()
               )

# Read PromotionUUID from Dim_Promotion
DF_Promotion = spark.read.format('delta').load(SilverFilePath_Promotion).filter('current_date() >= EffectiveDate and current_date() <= TerminationDate').select('PromotionUUID','PromotionName')

In [0]:
#Eligible MaterialCd
df_MaterialCd = (spark.read.options(header='true', delimiter=',').csv(RawMaterialCodeFilePath)
                           .withColumn("EffectiveDate", to_date('EffectiveDate', "MM/dd/yyyy"))
                           .withColumn("TerminationDate", to_date('TerminationDate', "MM/dd/yyyy")))

df_MaterialCd = df_MaterialCd.join(DF_Promotion,'PromotionName','inner')

df_MaterialCd.display()

df_MaterialCd_Eligible = df_MaterialCd.filter('current_date() >= EffectiveDate and current_date() <= TerminationDate')
df_MaterialCd_Terminated = df_MaterialCd.filter('current_date() > TerminationDate')

display(df_MaterialCd_Eligible)

In [0]:
# #Terminated Smart Cards
# df_SmartCard_Terminated  = df_EquipmentMaster.join(broadcast(df_MaterialCd_Eligible),'MaterialCd','inner')\
#                                            .select('SerialNumberNbr',
#                                                    'MaterialCd',
#                                                    'EffectiveDate',
#                                                    'TerminationDate',
#                                                    'Active')\
#                                             .withColumn('Comments',lit('EquipmentMaster'))\

# display(df_SmartCard_Terminated)

#Creating Log Entry
log_dict = {
"ConfigID" : ConfigID,
"SourceTypeID" : sourceTypeId,
"Source" : "EquipmentMaster",
"SourceFileName" : "",
"Destination" : "promotion.dim_smartcard",
"DestinationFileName" : "",
"Run_ID": str(PipelineRunId),
"Job_ID": str(JobId)
}
audit_df = spark.createDataFrame([log_dict])

In [0]:
try:
    #Eligible Smart Cards
    df_SmartCard_Eligible  = df_EquipmentMaster.join(broadcast(df_MaterialCd_Eligible),'MaterialCd','inner')\
                                            .select('SerialNumberNbr',
                                                    'MaterialCd',
                                                    'PromotionUUID',
                                                    'EffectiveDate',
                                                    'TerminationDate',
                                                    'Active')\
                                                .withColumn('Comments',lit('SmartCardMaterialCd'))

    df_smartCard = df_SmartCard_Eligible.join(df_ProdHrchy,"MaterialCd","left")\
                                .withColumn('SmartCardUUID',expr('uuid()'))\
                                .withColumnRenamed('SerialNumberNbr','SmartCardSerialNumber')\
                                .withColumn('CreatedBy',lit('ADB_SmartCard'))\
                                .withColumn('CreatedDate',current_timestamp())\
                                .withColumn('UpdatedBy',lit('ADB_SmartCard'))\
                                .withColumn('UpdatedDate',current_timestamp())\
                                .select('SmartCardUUID','SmartCardSerialNumber','MaterialCd','MaterialDescription',
                                        'EffectiveDate','TerminationDate','PromotionUUID','Comments',
                                        'CreatedDate','CreatedBy','UpdatedDate','UpdatedBy','Active')

    df_smartCard_dropduplicates = df_smartCard.withColumn("Rnk",row_number().over(Window.partitionBy("SmartCardSerialNumber","PromotionUUID").orderBy(col("EffectiveDate").asc())))\
                                                .where("rnk==1")\
                                                .drop("Rnk")

    df_SmartCard_Hist = DeltaTable.forPath(spark, SilverFilePath_Smartcard)  
    (df_SmartCard_Hist.alias("tgt")
            .merge(
            df_smartCard_dropduplicates.alias("src"),
                "tgt.SmartCardSerialNumber = src.SmartCardSerialNumber and tgt.PromotionUUID = src.PromotionUUID and tgt.Active= 1")
        .whenMatchedUpdate(
            set = {
                "tgt.MaterialDescription": "src.MaterialDescription",
                "tgt.EffectiveDate": "src.EffectiveDate",
                "tgt.TerminationDate": "src.TerminationDate",
                "tgt.MaterialCd" : "src.MaterialCd",
                "tgt.Comments": "src.Comments",
                "tgt.Active": "src.Active",
                "tgt.UpdatedDate": "src.UpdatedDate",
                "tgt.UpdatedBy": "src.UpdatedBy"
                }
        )
        .whenNotMatchedInsertAll()
        .execute())
    logIntoPromotionLogTable(audit_df,CreatedBy,"Succeeded")
except Exception as e:
    logIntoPromotionLogTable(audit_df,CreatedBy,"Failed",str(e))
    raise e

In [0]:
display(df_smartCard_dropduplicates)